In [4]:
import pandas as pd
import boto3
from io import BytesIO

# Load merged data from S3
s3 = boto3.client('s3')
bucket = 'nba-betting-mt'
key = 'data/03_intermediate/player_props_with_actuals_2025-26.csv'

obj = s3.get_object(Bucket=bucket, Key=key)
df = pd.read_csv(BytesIO(obj['Body'].read()))

print(f"✅ Loaded {len(df):,} rows, {len(df.columns)} columns")
print(f"Date range: {df['game_date'].min()} to {df['game_date'].max()}")

# Check scorer type columns
print(f"\nScorer type columns:")
print(f"  pts_0_6_pct: {df['pts_0_6_pct'].describe()}")
print(f"\nScorer type distribution:")
print(df[['PLAYER_NAME', 'scorer_type', 'pts_0_6_pct']].drop_duplicates('PLAYER_NAME')['scorer_type'].value_counts())

# Show sample
df[['PLAYER_NAME', 'game_date', 'PTS', 'points_line', 'pts_0_6_pct', 'scorer_type']].head(10)

✅ Loaded 11,691 rows, 49 columns
Date range: 2025-10-21 to 2026-01-05

Scorer type columns:
  pts_0_6_pct: count    11330.000000
mean        36.311114
std         18.821249
min          0.000000
25%         21.875000
50%         33.913043
75%         47.761194
max        100.000000
Name: pts_0_6_pct, dtype: float64

Scorer type distribution:
scorer_type
Perimeter (<50%)       358
Rim Attacker (≥50%)    127
Name: count, dtype: int64


,PLAYER_NAME,game_date,PTS,points_line,pts_0_6_pct,scorer_type
0,Luka Dončić,2025-10-21,43,30.700000,16.685714,Perimeter (<50%)
1,Alperen Sengun,2025-10-21,39,18.323529,48.524590,Perimeter (<50%)
2,Shai Gilgeous-Alexander,2025-10-21,35,32.200000,25.528169,Perimeter (<50%)
3,Austin Reaves,2025-10-21,26,22.055556,28.104575,Perimeter (<50%)
4,Jimmy Butler III,2025-10-21,31,17.264706,40.699523,Perimeter (<50%)
5,Stephen Curry,2025-10-21,23,26.261905,19.870968,Perimeter (<50%)
6,Chet Holmgren,2025-10-21,28,16.794118,42.560554,Perimeter (<50%)
7,Jonathan Kuminga,2025-10-21,17,14.038462,48.113208,Perimeter (<50%)
8,Cason Wallace,2025-10-21,14,7.500000,34.558824,Perimeter (<50%)
9,Kevin Durant,2025-10-21,23,24.131579,14.805521,Perimeter (<50%)


In [5]:
# Show examples of each scorer type
print("="*80)
print("SCORER TYPE EXAMPLES")
print("="*80)

# Get unique players with their scorer type
players = df[['PLAYER_NAME', 'scorer_type', 'pts_0_6_pct', 'rim_season_points', 'total_pts_season']].drop_duplicates('PLAYER_NAME')

# Rim Attackers (≥50% from 0-6 feet)
print("\n🔥 RIM ATTACKERS (≥50% of points from 0-6 feet):")
print("-"*80)
rim_attackers = players[players['scorer_type'] == 'Rim Attacker (≥50%)'].sort_values('pts_0_6_pct', ascending=False)
print(rim_attackers.head(15).to_string(index=False))

# Perimeter Players (<50% from 0-6 feet)
print("\n\n🎯 PERIMETER PLAYERS (<50% of points from 0-6 feet):")
print("-"*80)
perimeter = players[players['scorer_type'] == 'Perimeter (<50%)'].sort_values('pts_0_6_pct', ascending=False)
print(perimeter.head(15).to_string(index=False))

# Most extreme examples
print("\n\n📊 MOST EXTREME EXAMPLES:")
print("-"*80)
print(f"\nMost rim-dependent (highest % from 0-6 ft):")
print(players.nlargest(5, 'pts_0_6_pct')[['PLAYER_NAME', 'pts_0_6_pct', 'rim_season_points', 'total_pts_season']].to_string(index=False))

print(f"\nMost perimeter-oriented (lowest % from 0-6 ft):")
print(players.nsmallest(5, 'pts_0_6_pct')[['PLAYER_NAME', 'pts_0_6_pct', 'rim_season_points', 'total_pts_season']].to_string(index=False))

# Right at the 50% cutoff
print(f"\nPlayers near the 50% cutoff:")
near_cutoff = players[(players['pts_0_6_pct'] >= 48) & (players['pts_0_6_pct'] <= 52)].sort_values('pts_0_6_pct', ascending=False)
print(near_cutoff[['PLAYER_NAME', 'pts_0_6_pct', 'scorer_type']].to_string(index=False))

SCORER TYPE EXAMPLES

🔥 RIM ATTACKERS (≥50% of points from 0-6 feet):
--------------------------------------------------------------------------------
      PLAYER_NAME         scorer_type  pts_0_6_pct  rim_season_points  total_pts_season
   Oscar Tshiebwe Rim Attacker (≥50%)   100.000000                2.0               2.0
  Trentyn Flowers Rim Attacker (≥50%)   100.000000                4.0               4.0
Wendell Moore Jr. Rim Attacker (≥50%)   100.000000                6.0               6.0
  Lachlan Olbrich Rim Attacker (≥50%)   100.000000                2.0               2.0
     Curtis Jones Rim Attacker (≥50%)   100.000000                4.0               4.0
      Colby Jones Rim Attacker (≥50%)   100.000000                2.0               2.0
  Daeqwon Plowden Rim Attacker (≥50%)   100.000000                2.0               2.0
    James Wiseman Rim Attacker (≥50%)    92.307692               12.0              13.0
  Bismack Biyombo Rim Attacker (≥50%)    88.888889       

In [6]:
# at least 5ppg...
# ===============
# Show examples of each scorer type (min 5 PPG)
print("="*80)
print("SCORER TYPE EXAMPLES (≥5 PPG)")
print("="*80)

# Get unique players with their scorer type and calculate PPG
players = df.groupby('PLAYER_NAME').agg({
    'scorer_type': 'first',
    'pts_0_6_pct': 'first',
    'rim_season_points': 'first',
    'total_pts_season': 'first',
    'PTS': 'mean'  # Average PPG
}).reset_index()
players.columns = ['PLAYER_NAME', 'scorer_type', 'pts_0_6_pct', 'rim_season_points', 'total_pts_season', 'avg_ppg']

# Filter to players averaging at least 5 PPG
players = players[players['avg_ppg'] >= 5.0]

print(f"\nTotal players ≥5 PPG: {len(players)}")
print(f"  Rim Attackers: {(players['scorer_type'] == 'Rim Attacker (≥50%)').sum()}")
print(f"  Perimeter: {(players['scorer_type'] == 'Perimeter (<50%)').sum()}")

# Rim Attackers (≥50% from 0-6 feet)
print("\n🔥 RIM ATTACKERS (≥50% of points from 0-6 feet, ≥5 PPG):")
print("-"*80)
rim_attackers = players[players['scorer_type'] == 'Rim Attacker (≥50%)'].sort_values('pts_0_6_pct', ascending=False)
print(rim_attackers[['PLAYER_NAME', 'avg_ppg', 'pts_0_6_pct', 'rim_season_points', 'total_pts_season']].head(15).to_string(index=False))

# Perimeter Players (<50% from 0-6 feet)
print("\n\n🎯 PERIMETER PLAYERS (<50% of points from 0-6 feet, ≥5 PPG):")
print("-"*80)
perimeter = players[players['scorer_type'] == 'Perimeter (<50%)'].sort_values('pts_0_6_pct', ascending=False)
print(perimeter[['PLAYER_NAME', 'avg_ppg', 'pts_0_6_pct', 'rim_season_points', 'total_pts_season']].head(15).to_string(index=False))

# Most extreme examples
print("\n\n📊 MOST EXTREME EXAMPLES (≥5 PPG):")
print("-"*80)
print(f"\nMost rim-dependent (highest % from 0-6 ft):")
print(players.nlargest(10, 'pts_0_6_pct')[['PLAYER_NAME', 'avg_ppg', 'pts_0_6_pct']].to_string(index=False))

print(f"\nMost perimeter-oriented (lowest % from 0-6 ft):")
print(players.nsmallest(10, 'pts_0_6_pct')[['PLAYER_NAME', 'avg_ppg', 'pts_0_6_pct']].to_string(index=False))

# Right at the 50% cutoff
print(f"\nPlayers near the 50% cutoff:")
near_cutoff = players[(players['pts_0_6_pct'] >= 48) & (players['pts_0_6_pct'] <= 52)].sort_values('pts_0_6_pct', ascending=False)
print(near_cutoff[['PLAYER_NAME', 'avg_ppg', 'pts_0_6_pct', 'scorer_type']].to_string(index=False))

SCORER TYPE EXAMPLES (≥5 PPG)

Total players ≥5 PPG: 330
  Rim Attackers: 62
  Perimeter: 255

🔥 RIM ATTACKERS (≥50% of points from 0-6 feet, ≥5 PPG):
--------------------------------------------------------------------------------
        PLAYER_NAME   avg_ppg  pts_0_6_pct  rim_season_points  total_pts_season
Robert Williams III  5.875000    85.106383              120.0             141.0
   Ryan Kalkbrenner  8.692308    82.300885              186.0             226.0
         Yves Missi  5.172414    81.333333              122.0             150.0
       Jaxson Hayes  6.178571    78.612717              136.0             173.0
      Neemias Queta 10.151515    78.208955              262.0             335.0
       Goga Bitadze  5.875000    76.595745              144.0             188.0
        Rudy Gobert 11.277778    76.354680              310.0             406.0
     Isaiah Jackson  7.310345    75.471698              160.0             212.0
     Daniel Gafford  8.200000    73.170732      